# Day 06 - Aggregations and Joins

**Topic:** Aggregations and Joins   
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกใช้ `groupBy`, `agg`, และ join types ที่ใช้บ่อยในงาน Data Engineering เช่น `inner`, `left`, `left_anti`, และ `left_semi`

วันนี้เราจะเรียนการสรุปข้อมูลด้วย aggregation และการเชื่อมข้อมูลหลาย DataFrame ด้วย join ใน PySpark ผ่านตัวอย่างใกล้เคียงงาน Data Engineering จริง เช่น transaction summary, customer enrichment, unmatched record validation และ active customer filtering

## Goal of the day

1. Aggregate transaction data ด้วย `groupBy()` และ `agg()`
2. ใช้ aggregate functions เช่น `count`, `sum`, `avg`, `min`, `max`, `countDistinct`
3. Join fact-like data กับ master data เช่น customer/product
4. เลือก join type ให้เหมาะกับ use case
5. ใช้ `left_anti` เพื่อหา unmatched records
6. ใช้ `left_semi` เพื่อ filter เฉพาะ records ที่มี key อยู่ใน reference/master table
7. เขียนผลลัพธ์ summary เป็น Lakehouse table ด้วย `saveAsTable()`

## Why it matters in production

ใน production pipeline งานส่วนใหญ่ไม่ได้จบที่การ clean column อย่างเดียว แต่ต้องรวมข้อมูลจากหลายแหล่งและสรุปออกมาเป็น table ที่ business ใช้ต่อได้

ตัวอย่าง use case จริง:

- สร้าง revenue summary จาก transaction data
- Join transaction กับ customer master เพื่อ enrich ข้อมูล
- Join product master เพื่อเติม category หรือ product name
- หา transaction ที่อ้างถึง customer/product ที่ไม่มีอยู่ใน master data
- Build Gold table สำหรับ dashboard หรือ reporting

Aggregation และ Join เป็นจุดที่ทำให้ Spark job แพงขึ้นได้ง่าย เพราะมักทำให้เกิด **shuffle** โดยเฉพาะ `groupBy` และ `join` บน dataset ขนาดใหญ่ หรือ key ที่มีจำนวน distinct values สูง (high-cardinality keys) วันนี้ยังไม่ deep เรื่อง performance แต่จะเริ่มฝึก mindset ว่าควรเลือก join type และ aggregation logic อย่างตั้งใจ

## Key concepts

| Concept | Meaning |
|---|---|
| `groupBy()` | จัดกลุ่ม rows ตาม column หนึ่งตัวหรือหลายตัว |
| `agg()` | ใช้ aggregate functions หลังจากจัดกลุ่มข้อมูลแล้ว |
| `count()` | นับจำนวน rows หรือจำนวนค่าที่ไม่เป็น null |
| `sum()` | รวมค่าของ numeric column |
| `avg()` | หาค่าเฉลี่ยของ numeric column |
| `min()` / `max()` | หาค่าต่ำสุด / สูงสุด |
| `countDistinct()` | นับจำนวนค่าที่ไม่ซ้ำ |
| `inner join` | เก็บเฉพาะ records ที่ match กันทั้งสองฝั่ง |
| `left join` | เก็บ records ทั้งหมดจากฝั่งซ้าย และเติม columns จากฝั่งขวาเมื่อ match |
| `left_anti join` | เก็บ records จากฝั่งซ้ายที่ไม่เจอ match ในฝั่งขวา |
| `left_semi join` | เก็บ records จากฝั่งซ้ายที่เจอ match ในฝั่งขวา แต่ไม่ดึง columns จากฝั่งขวามาเพิ่ม |

## Concept explanation

### Aggregation

Aggregation คือการสรุปหลาย records ให้เหลือ summary ในระดับที่ต้องการ เช่น:

- ยอดขายรวมต่อ customer
- จำนวน transaction ต่อ payment method
- average amount ต่อ region
- min/max transaction amount ต่อ product category

Pattern หลักใน PySpark คือ:

```python
df.groupBy("group_column").agg(
    F.count("*").alias("row_count"),
    F.sum("amount").alias("total_amount")
)
```

### Join

Join คือการรวม DataFrame มากกว่า 1 ตัวเข้าด้วยกันผ่าน key เช่น `customer_id` หรือ `product_id`

ใน Data Engineering มักมี pattern แบบนี้:

- Fact data: transaction, event, order
- Master/reference data: customer, product, property, branch

สิ่งที่ต้องคิดเสมอ:

1. Join key ถูกต้องไหม
2. Master table มี duplicate key ไหม
3. ต้องการเก็บ unmatched records หรือไม่
4. Join แล้วจำนวน rows เพิ่มผิดปกติไหม
5. Aggregate ก่อน join หรือ join ก่อน aggregate เหมาะกว่ากัน
    - ถ้า aggregate ก่อนได้ มักช่วยลดจำนวน rows ก่อน join และลด shuffle เช่น สรุปยอด transaction ต่อ customer_id ก่อน แล้วค่อย join กับ customer master
    - แต่ถ้าต้องใช้ข้อมูลจาก master table ในการจัดกลุ่ม ต้อง join ก่อนแล้วค่อย aggregate เช่น ต้อง join เพื่อเอา region มาก่อน แล้วค่อยสรุปยอดขายตาม region

### Join type mindset

| Join type | ใช้เมื่อไร |
|---|---|
| `inner` | ต้องการเฉพาะ records ที่ match ทั้งสองฝั่ง |
| `left` | ต้องการเก็บ fact records ทั้งหมดไว้ แม้ master จะ missing |
| `left_anti` | ต้องการหา records ที่ไม่มี reference เช่น unknown customer |
| `left_semi` | ต้องการ filter เฉพาะ records ที่มี reference โดยไม่ต้องดึง column จากขวา |


## Mermaid diagram: Aggregation and Join Flow

```mermaid
flowchart LR
    T[Transactions<br/>fact-like data] --> A[Aggregate<br/>groupBy + agg]
    T --> J1[Join customer master]
    C[Customers<br/>master data] --> J1

    J1 --> J2[Join product master]
    P[Products<br/>reference data] --> J2

    J2 --> G[Gold-style summary<br/>customer revenue]

    T --> V1[left_anti join]
    C --> V1
    V1 --> U[Unknown customer records]

    T --> V2[left_semi join]
    C --> V2
    V2 --> Valid[Valid customer transactions]
```

ภาพนี้สรุป flow หลักของวันนี้: transaction data สามารถถูก aggregate เพื่อทำ summary, join เพื่อ enrich, หรือใช้ anti/semi join เพื่อ validate reference key ได้

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
)

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 3, Finished, Available, Finished, False)

## Create mock data

Dataset มี 3 ชุด:

1. `df_transactions` - fact-like transaction data
2. `df_customers` - customer master
3. `df_products` - product reference

ตั้งใจใส่ unmatched key ไว้ด้วย:

- `customer_id = C999` ไม่มีใน customer master
- `product_id = P999` ไม่มีใน product master

เพื่อใช้ฝึก `left_anti join`

In [2]:
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True),
])

transactions_data = [
    ("T001", "C001", "P001", "2026-05-01", 1250.00, "credit_card", "online", "success"),
    ("T002", "C002", "P002", "2026-05-01", 2990.00, "promptpay", "online", "success"),
    ("T003", "C001", "P001", "2026-05-02", 0.00, "credit_card", "online", "failed"),
    ("T004", "C003", "P003", "2026-05-02", 4000.00, "bank_transfer", "branch", "success"),
    ("T005", "C005", "P004", "2026-05-03", 560.00, "promptpay", "online", "pending"),
    ("T006", "C004", "P001", "2026-05-03", 780.00, "credit_card", "branch", "success"),
    ("T007", "C999", "P002", "2026-05-04", 650.00, "promptpay", "online", "success"),
    ("T008", "C001", "P999", "2026-05-04", 1200.00, "credit_card", "online", "success"),
]

df_transactions_raw = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions = df_transactions_raw.withColumn(
    "transaction_date",
    F.to_date("transaction_date")
)

df_transactions.show()
df_transactions.printSchema()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 4, Finished, Available, Finished, False)

+--------------+-----------+----------+----------------+------+--------------+-------+-------+
|transaction_id|customer_id|product_id|transaction_date|amount|payment_method|channel| status|
+--------------+-----------+----------+----------------+------+--------------+-------+-------+
|          T001|       C001|      P001|      2026-05-01|1250.0|   credit_card| online|success|
|          T002|       C002|      P002|      2026-05-01|2990.0|     promptpay| online|success|
|          T003|       C001|      P001|      2026-05-02|   0.0|   credit_card| online| failed|
|          T004|       C003|      P003|      2026-05-02|4000.0| bank_transfer| branch|success|
|          T005|       C005|      P004|      2026-05-03| 560.0|     promptpay| online|pending|
|          T006|       C004|      P001|      2026-05-03| 780.0|   credit_card| branch|success|
|          T007|       C999|      P002|      2026-05-04| 650.0|     promptpay| online|success|
|          T008|       C001|      P999|      2026-

In [3]:
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("region", StringType(), True),
    StructField("status", StringType(), True),
])

customers_data = [
    ("C001", "Alice", "retail", "Bangkok", "active"),
    ("C002", "Bob", "retail", "Chiang Mai", "active"),
    ("C003", "Charlie", "corporate", "Bangkok", "active"),
    ("C004", "Darin", "retail", "Khon Kaen", "inactive"),
    ("C005", "Eve", "corporate", "Phuket", "active"),
]

df_customers = spark.createDataFrame(customers_data, customer_schema)

df_customers.show()
df_customers.printSchema()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 6, Finished, Available, Finished, False)

+-----------+-------------+---------+----------+--------+
|customer_id|customer_name|  segment|    region|  status|
+-----------+-------------+---------+----------+--------+
|       C001|        Alice|   retail|   Bangkok|  active|
|       C002|          Bob|   retail|Chiang Mai|  active|
|       C003|      Charlie|corporate|   Bangkok|  active|
|       C004|        Darin|   retail| Khon Kaen|inactive|
|       C005|          Eve|corporate|    Phuket|  active|
+-----------+-------------+---------+----------+--------+

root
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)



In [4]:
product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("list_price", DoubleType(), True),
])

products_data = [
    ("P001", "Starter Package", "package", 1200.00),
    ("P002", "Premium Package", "package", 3000.00),
    ("P003", "Consulting Service", "service", 4000.00),
    ("P004", "Support Add-on", "service", 600.00),
]

df_products = spark.createDataFrame(products_data, product_schema)

df_products.show()
df_products.printSchema()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 8, Finished, Available, Finished, False)

+----------+------------------+--------+----------+
|product_id|      product_name|category|list_price|
+----------+------------------+--------+----------+
|      P001|   Starter Package| package|    1200.0|
|      P002|   Premium Package| package|    3000.0|
|      P003|Consulting Service| service|    4000.0|
|      P004|    Support Add-on| service|     600.0|
+----------+------------------+--------+----------+

root
 |-- product_id: string (nullable = false)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: double (nullable = true)



## PySpark code examples

### Example 1: Basic aggregation by customer

เริ่มจาก filter เฉพาะ `success` transactions ก่อน aggregate

เหตุผล: ใน production ถ้าไม่ filter status ให้ชัด อาจเผลอเอา failed/pending transaction ไปคิดยอด revenue

In [5]:
df_success_transactions = df_transactions.filter(F.col("status") == "success")

df_customer_transaction_summary = (
    df_success_transactions
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
        F.min("amount").alias("min_amount"),
        F.max("amount").alias("max_amount"),
    )
    .orderBy(F.desc("total_amount"))
)

df_customer_transaction_summary.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 10, Finished, Available, Finished, False)

+-----------+-----------------+------------+----------+----------+----------+
|customer_id|transaction_count|total_amount|avg_amount|min_amount|max_amount|
+-----------+-----------------+------------+----------+----------+----------+
|       C003|                1|      4000.0|    4000.0|    4000.0|    4000.0|
|       C002|                1|      2990.0|    2990.0|    2990.0|    2990.0|
|       C001|                2|      2450.0|    1225.0|    1200.0|    1250.0|
|       C004|                1|       780.0|     780.0|     780.0|     780.0|
|       C999|                1|       650.0|     650.0|     650.0|     650.0|
+-----------+-----------------+------------+----------+----------+----------+



### Example 2: Aggregation by multiple columns

`groupBy()` สามารถ group หลาย columns ได้ เช่น สรุปยอดตาม `transaction_date` และ `channel`

Pattern นี้ใช้บ่อยเวลาทำ daily summary, channel summary, source system summary

In [6]:
df_daily_channel_summary = (
    df_success_transactions
    .groupBy("transaction_date", "channel")
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("customer_id").alias("distinct_customer_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
    )
    .orderBy("transaction_date", "channel")
)

df_daily_channel_summary.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 13, Finished, Available, Finished, False)

+----------------+-------+-----------------+-----------------------+------------+
|transaction_date|channel|transaction_count|distinct_customer_count|total_amount|
+----------------+-------+-----------------+-----------------------+------------+
|      2026-05-01| online|                2|                      2|      4240.0|
|      2026-05-02| branch|                1|                      1|      4000.0|
|      2026-05-03| branch|                1|                      1|       780.0|
|      2026-05-04| online|                2|                      2|      1850.0|
+----------------+-------+-----------------+-----------------------+------------+



### Example 3: Use `explain()` to inspect aggregation plan

Aggregation มักเป็น **wide transformation** เพราะ Spark ต้อง group records ที่มี key เดียวกันเข้าด้วยกัน ซึ่งมักเกี่ยวข้องกับ shuffle

- **Wide transformation** คือ operation ที่มักต้องย้ายหรือเทียบข้อมูลข้าม partitions เช่น `groupBy`, `join`, `distinct`, `dropDuplicates` ทำให้มักแพงกว่า **narrow transformation** ที่ประมวลผลต่อ partition เดิมได้ เช่น `select`, `filter`, `withColumn`

ตอนนี้ยังไม่ต้องอ่าน plan ลึก แค่เริ่มสังเกตคำอย่าง `Exchange`, `HashAggregate`, หรือ `SortAggregate`

In [7]:
df_customer_transaction_summary.explain(mode="formatted")

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 15, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, product_id#728, transaction_date#729, amount#730, payment_method#731, channel#732, status#733]
Arguments: [transaction_id#726, customer_id#727, product_id#728, transaction_date#729, amount#730, payment_method#731, channel#732, status#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [8]: [transaction_id#726, customer_id#727, product_id#728, transaction_date#729, amount#730, payment_method#731, channel#732, status#733]
Condition : (isnotnull(status#733) AND (status#733 = success))

(3) Project
Output [2]: [customer_id#727, amount#730]
Input [8]: [transaction_id#72

### Example 4: Inner join transaction with customer master

`inner join` จะเก็บเฉพาะ records ที่ match ทั้งสองฝั่ง

ในตัวอย่างนี้ transaction ที่ `customer_id = C999` จะหายไป เพราะไม่มี customer master

In [10]:
df_transactions_with_customer_inner = (
    df_success_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="inner")
    .select(
        "transaction_id",
        "customer_id",
        "customer_name",
        "segment",
        "region",
        "product_id",
        "transaction_date",
        "amount",
        "payment_method",
        "channel",
    )
    .orderBy("transaction_id")
)

# Tip:
# In production joins, prefer F.col("t.column") / F.col("c.column") when aliases are used.
# This makes column origin explicit and helps avoid duplicate or ambiguous column issues.

df_transactions_with_customer_inner.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 25, Finished, Available, Finished, False)

+--------------+-----------+-------------+---------+----------+----------+----------------+------+--------------+-------+
|transaction_id|customer_id|customer_name|  segment|    region|product_id|transaction_date|amount|payment_method|channel|
+--------------+-----------+-------------+---------+----------+----------+----------------+------+--------------+-------+
|          T001|       C001|        Alice|   retail|   Bangkok|      P001|      2026-05-01|1250.0|   credit_card| online|
|          T002|       C002|          Bob|   retail|Chiang Mai|      P002|      2026-05-01|2990.0|     promptpay| online|
|          T004|       C003|      Charlie|corporate|   Bangkok|      P003|      2026-05-02|4000.0| bank_transfer| branch|
|          T006|       C004|        Darin|   retail| Khon Kaen|      P001|      2026-05-03| 780.0|   credit_card| branch|
|          T008|       C001|        Alice|   retail|   Bangkok|      P999|      2026-05-04|1200.0|   credit_card| online|
+--------------+--------

### Example 5: Left join transaction with product reference

`left join` จะเก็บ records ฝั่งซ้ายไว้ทั้งหมด แล้วเติมข้อมูลจากฝั่งขวาเท่าที่ match ได้

ในตัวอย่างนี้ transaction ที่ `product_id = P999` ยังอยู่ แต่ `product_name`, `category`, `list_price` จะเป็น `null`

Pattern นี้เหมาะมากเวลาต้องการ preserve fact records เพื่อ debug ต่อ

In [11]:
df_transactions_with_product_left = (
    df_success_transactions.alias("t")
    .join(df_products.alias("p"), on="product_id", how="left")
    .select(
        "transaction_id",
        "customer_id",
        "product_id",
        "product_name",
        "category",
        "list_price",
        "transaction_date",
        "amount",
        "status",
    )
    .orderBy("transaction_id")
)

df_transactions_with_product_left.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 54, Finished, Available, Finished, False)

+--------------+-----------+----------+------------------+--------+----------+----------------+------+-------+
|transaction_id|customer_id|product_id|      product_name|category|list_price|transaction_date|amount| status|
+--------------+-----------+----------+------------------+--------+----------+----------------+------+-------+
|          T001|       C001|      P001|   Starter Package| package|    1200.0|      2026-05-01|1250.0|success|
|          T002|       C002|      P002|   Premium Package| package|    3000.0|      2026-05-01|2990.0|success|
|          T004|       C003|      P003|Consulting Service| service|    4000.0|      2026-05-02|4000.0|success|
|          T006|       C004|      P001|   Starter Package| package|    1200.0|      2026-05-03| 780.0|success|
|          T007|       C999|      P002|   Premium Package| package|    3000.0|      2026-05-04| 650.0|success|
|          T008|       C001|      P999|              NULL|    NULL|      NULL|      2026-05-04|1200.0|success|
+

### Example 6: Left anti join to find unknown customer records

`left_anti join` ใช้หา records ฝั่งซ้ายที่ **ไม่มี match** ในฝั่งขวา

ใช้บ่อยในงาน validation เช่น:

- transaction มี customer_id ที่ไม่มีใน customer master
- order มี product_id ที่ไม่มีใน product master
- fact table มี foreign key ที่ไม่ match dimension table

In [12]:
df_unknown_customer_transactions = (
    df_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="left_anti")
    .orderBy("transaction_id")
)

df_unknown_customer_transactions.show()
print("Unknown customer transaction count:", df_unknown_customer_transactions.count())

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 56, Finished, Available, Finished, False)

+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|customer_id|transaction_id|product_id|transaction_date|amount|payment_method|channel| status|
+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|       C999|          T007|      P002|      2026-05-04| 650.0|     promptpay| online|success|
+-----------+--------------+----------+----------------+------+--------------+-------+-------+

Unknown customer transaction count: 1


### Example 7: Left anti join to find unknown product records

ใช้ logic เดียวกันเพื่อหา transaction ที่อ้างถึง product ที่ไม่มีใน product reference

In [13]:
df_unknown_product_transactions = (
    df_transactions.alias("t")
    .join(df_products.alias("p"), on="product_id", how="left_anti")
    .orderBy("transaction_id")
)

df_unknown_product_transactions.show()
print("Unknown product transaction count:", df_unknown_product_transactions.count())

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 59, Finished, Available, Finished, False)

+----------+--------------+-----------+----------------+------+--------------+-------+-------+
|product_id|transaction_id|customer_id|transaction_date|amount|payment_method|channel| status|
+----------+--------------+-----------+----------------+------+--------------+-------+-------+
|      P999|          T008|       C001|      2026-05-04|1200.0|   credit_card| online|success|
+----------+--------------+-----------+----------------+------+--------------+-------+-------+

Unknown product transaction count: 1


### Example 8: Left semi join to keep only records with valid customers

`left_semi join` จะ return เฉพาะ columns จากฝั่งซ้าย แต่ filter เฉพาะ rows ที่มี key match กับฝั่งขวา

ต่างจาก `inner join` ตรงที่ `left_semi` ไม่เอา columns จาก master table เข้ามา

เหมาะกับ use case เช่น:

- keep เฉพาะ transaction ที่มี customer master
- filter source records ด้วย allowlist/reference table
- ลดความเสี่ยง duplicate จาก master table ที่มีหลาย rows ต่อ key

In [14]:
df_valid_customer_transactions = (
    df_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="left_semi")
    .orderBy("transaction_id")
)

df_valid_customer_transactions.show()
print("Valid customer transaction count:", df_valid_customer_transactions.count())

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 64, Finished, Available, Finished, False)

+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|customer_id|transaction_id|product_id|transaction_date|amount|payment_method|channel| status|
+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|       C001|          T001|      P001|      2026-05-01|1250.0|   credit_card| online|success|
|       C002|          T002|      P002|      2026-05-01|2990.0|     promptpay| online|success|
|       C001|          T003|      P001|      2026-05-02|   0.0|   credit_card| online| failed|
|       C003|          T004|      P003|      2026-05-02|4000.0| bank_transfer| branch|success|
|       C005|          T005|      P004|      2026-05-03| 560.0|     promptpay| online|pending|
|       C004|          T006|      P001|      2026-05-03| 780.0|   credit_card| branch|success|
|       C001|          T008|      P999|      2026-05-04|1200.0|   credit_card| online|success|
+-----------+--------------+----------+-----------

### Example 9: Build enriched transaction DataFrame

รวม transaction + customer + product เพื่อสร้าง enriched dataset

ใช้ `left join` กับ product เพื่อไม่ให้ transaction หายถ้า product reference missing และสามารถเห็นปัญหาเป็น `null` ได้

In [15]:
df_enriched_success_transactions = (
    df_success_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="inner")
    .join(df_products.alias("p"), on="product_id", how="left")
    .select(
        "transaction_id",
        "transaction_date",
        "customer_id",
        "customer_name",
        "segment",
        "region",
        "product_id",
        "product_name",
        "category",
        "amount",
        "payment_method",
        "channel",
    )
    .orderBy("transaction_id")
)

df_enriched_success_transactions.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 66, Finished, Available, Finished, False)

+--------------+----------------+-----------+-------------+---------+----------+----------+------------------+--------+------+--------------+-------+
|transaction_id|transaction_date|customer_id|customer_name|  segment|    region|product_id|      product_name|category|amount|payment_method|channel|
+--------------+----------------+-----------+-------------+---------+----------+----------+------------------+--------+------+--------------+-------+
|          T001|      2026-05-01|       C001|        Alice|   retail|   Bangkok|      P001|   Starter Package| package|1250.0|   credit_card| online|
|          T002|      2026-05-01|       C002|          Bob|   retail|Chiang Mai|      P002|   Premium Package| package|2990.0|     promptpay| online|
|          T004|      2026-05-02|       C003|      Charlie|corporate|   Bangkok|      P003|Consulting Service| service|4000.0| bank_transfer| branch|
|          T006|      2026-05-03|       C004|        Darin|   retail| Khon Kaen|      P001|   Starte

### Example 10: Build Gold-style customer revenue summary

หลังจาก join แล้ว เราสามารถ aggregate เพื่อสร้าง summary table ได้

ตัวอย่างนี้ใกล้เคียง Gold table แบบง่าย ๆ สำหรับ dashboard/reporting

In [16]:
df_gold_customer_revenue = (
    df_enriched_success_transactions
    .groupBy("customer_id", "customer_name", "segment", "region")
    .agg(
        F.countDistinct("transaction_id").alias("success_transaction_count"),
        F.countDistinct("product_id").alias("distinct_product_count"),
        F.round(F.sum("amount"), 2).alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("avg_transaction_amount"),
    )
    .withColumn(
        "revenue_tier",
        F.when(F.col("total_revenue") >= 3000, F.lit("high"))
         .when(F.col("total_revenue") >= 1000, F.lit("medium"))
         .otherwise(F.lit("low"))
    )
    .orderBy(F.desc("total_revenue"))
)

df_gold_customer_revenue.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 68, Finished, Available, Finished, False)

+-----------+-------------+---------+----------+-------------------------+----------------------+-------------+----------------------+------------+
|customer_id|customer_name|  segment|    region|success_transaction_count|distinct_product_count|total_revenue|avg_transaction_amount|revenue_tier|
+-----------+-------------+---------+----------+-------------------------+----------------------+-------------+----------------------+------------+
|       C003|      Charlie|corporate|   Bangkok|                        1|                     1|       4000.0|                4000.0|        high|
|       C002|          Bob|   retail|Chiang Mai|                        1|                     1|       2990.0|                2990.0|      medium|
|       C001|        Alice|   retail|   Bangkok|                        2|                     2|       2450.0|                1225.0|      medium|
|       C004|        Darin|   retail| Khon Kaen|                        1|                     1|        780.0| 

### Example 11: Write summary to Lakehouse table

ใน Fabric Notebook ถ้า notebook attach กับ Lakehouse แล้ว สามารถเขียน table ด้วย `saveAsTable()` ได้

ถ้ายังไม่ได้ attach Lakehouse cell นี้อาจ fail ได้ ให้ attach Lakehouse ก่อน หรือข้าม cell นี้ชั่วคราว

In [17]:
table_name = "day06_customer_revenue_summary"

(
    df_gold_customer_revenue
    .write
    .mode("overwrite")
    .saveAsTable(table_name)
)

print(f"Saved table: {table_name}")

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 84, Finished, Available, Finished, False)

Saved table: day06_customer_revenue_summary


### Example 12: Read Lakehouse table back

หลังเขียน table แล้ว อ่านกลับด้วย `spark.read.table()` เพื่อ confirm ว่าผลลัพธ์ถูก save แล้ว

In [18]:
df_saved_customer_revenue = spark.read.table("day06_customer_revenue_summary")

df_saved_customer_revenue.show()
df_saved_customer_revenue.printSchema()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 85, Finished, Available, Finished, False)

+-----------+-------------+---------+----------+-------------------------+----------------------+-------------+----------------------+------------+
|customer_id|customer_name|  segment|    region|success_transaction_count|distinct_product_count|total_revenue|avg_transaction_amount|revenue_tier|
+-----------+-------------+---------+----------+-------------------------+----------------------+-------------+----------------------+------------+
|       C003|      Charlie|corporate|   Bangkok|                        1|                     1|       4000.0|                4000.0|        high|
|       C002|          Bob|   retail|Chiang Mai|                        1|                     1|       2990.0|                2990.0|      medium|
|       C001|        Alice|   retail|   Bangkok|                        2|                     2|       2450.0|                1225.0|      medium|
|       C004|        Darin|   retail| Khon Kaen|                        1|                     1|        780.0| 

## Quick recap

| Question | Answer |
|---|---|
| ใช้ `groupBy().agg()` เมื่อไร? | เมื่อต้องการสรุปข้อมูลหลาย rows ให้เป็น summary |
| `inner join` เหมาะกับอะไร? | เฉพาะ records ที่ต้อง match ทั้งสองฝั่งเท่านั้น |
| `left join` เหมาะกับอะไร? | เก็บ fact records ไว้ทั้งหมด และเติม master data ถ้ามี |
| `left_anti join` ใช้ทำอะไร? | หา records ที่ไม่มี reference/master match |
| `left_semi join` ต่างจาก inner อย่างไร? | filter เฉพาะ rows ที่ match แต่ไม่เพิ่ม columns จากฝั่งขวา |
| ทำไม aggregation/join อาจแพง? | มักเกิด shuffle เพราะต้องจัดกลุ่มหรือจับคู่ rows ตาม key |
| ทำไมต้องเช็ก row count หลัง join? | เพื่อจับปัญหา records หาย หรือ records เพิ่มจาก duplicate keys |

## Exercises

### Exercise 1: Aggregate successful transactions by channel

Requirements:

- ใช้เฉพาะ `status = success`
- Group by `channel`
- คำนวณ:
  - จำนวน transaction
  - จำนวน unique customer
  - total amount
  - average amount
- Sort จาก total amount มากไปน้อย

Expected result:

- เห็น revenue summary ต่อ channel
- Online ควรมียอดรวมสูงกว่า branch จาก mock data ชุดนี้

In [20]:
df_ex_channel_summary = (
    df_transactions
    .filter(F.col("status") == "success")
    .groupBy("channel")
    .agg(
        F.count("*").alias("transaction_count"),
        F.countDistinct("customer_id").alias("distinct_customer_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
    )
    .orderBy(F.desc("total_amount"))
)

df_ex_channel_summary.show()

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 95, Finished, Available, Finished, False)

+-------+-----------------+-----------------------+------------+----------+
|channel|transaction_count|distinct_customer_count|total_amount|avg_amount|
+-------+-----------------+-----------------------+------------+----------+
| online|                4|                      3|      6090.0|    1522.5|
| branch|                2|                      2|      4780.0|    2390.0|
+-------+-----------------+-----------------------+------------+----------+



### Exercise 2: Join transactions with customer and product data

Requirements:

- ใช้ `df_success_transactions`
- Join กับ `df_customers`
- Join กับ `df_products`
- แสดง columns หลัก:
  - transaction_id
  - customer_id
  - customer_name
  - product_id
  - product_name
  - category
  - amount

Expected result:

- Transaction ที่ customer unknown จะไม่อยู่ ถ้าใช้ `inner join` กับ customer
- Transaction ที่ product unknown ยังอยู่ ถ้าใช้ `left join` กับ product หลังจาก join customer แล้ว

In [21]:
df_ex_enriched_transactions = (
    df_success_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="inner")
    .join(df_products.alias("p"), on="product_id", how="left")
    .select(
        "transaction_id",
        "customer_id",
        "customer_name",
        "product_id",
        "product_name",
        "category",
        "amount",
    )
    .orderBy("transaction_id")
)

df_ex_enriched_transactions.show()
print("Enriched transaction count:", df_ex_enriched_transactions.count())

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 100, Finished, Available, Finished, False)

+--------------+-----------+-------------+----------+------------------+--------+------+
|transaction_id|customer_id|customer_name|product_id|      product_name|category|amount|
+--------------+-----------+-------------+----------+------------------+--------+------+
|          T001|       C001|        Alice|      P001|   Starter Package| package|1250.0|
|          T002|       C002|          Bob|      P002|   Premium Package| package|2990.0|
|          T004|       C003|      Charlie|      P003|Consulting Service| service|4000.0|
|          T006|       C004|        Darin|      P001|   Starter Package| package| 780.0|
|          T008|       C001|        Alice|      P999|              NULL|    NULL|1200.0|
+--------------+-----------+-------------+----------+------------------+--------+------+

Enriched transaction count: 5


### Exercise 3: Find unmatched reference records

Requirements:

- หา transactions ที่ไม่มี customer master ด้วย `left_anti`
- หา transactions ที่ไม่มี product reference ด้วย `left_anti`
- แสดง count ของแต่ละกลุ่ม

Expected result:

- Unknown customer transaction count = 1
- Unknown product transaction count = 1

In [22]:
df_ex_unknown_customers = (
    df_transactions.alias("t")
    .join(df_customers.alias("c"), on="customer_id", how="left_anti")
)

df_ex_unknown_products = (
    df_transactions.alias("t")
    .join(df_products.alias("p"), on="product_id", how="left_anti")
)

df_ex_unknown_customers.show()
df_ex_unknown_products.show()

print("Unknown customer transaction count:", df_ex_unknown_customers.count())
print("Unknown product transaction count:", df_ex_unknown_products.count())

StatementMeta(, 4f702837-36d5-42f3-a1de-02cf65d72dde, 109, Finished, Available, Finished, False)

+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|customer_id|transaction_id|product_id|transaction_date|amount|payment_method|channel| status|
+-----------+--------------+----------+----------------+------+--------------+-------+-------+
|       C999|          T007|      P002|      2026-05-04| 650.0|     promptpay| online|success|
+-----------+--------------+----------+----------------+------+--------------+-------+-------+

+----------+--------------+-----------+----------------+------+--------------+-------+-------+
|product_id|transaction_id|customer_id|transaction_date|amount|payment_method|channel| status|
+----------+--------------+-----------+----------------+------+--------------+-------+-------+
|      P999|          T008|       C001|      2026-05-04|1200.0|   credit_card| online|success|
+----------+--------------+-----------+----------------+------+--------------+-------+-------+

Unknown customer transaction count: 1
Unknown pr

## Common mistakes

1. **Aggregate โดยไม่ filter status ก่อน**

   ถ้าเอา `failed` หรือ `pending` transaction ไปคิด revenue อาจทำให้ตัวเลข business ผิด

2. **ใช้ `inner join` แล้ว records หายโดยไม่รู้ตัว**

   `inner join` จะตัด unmatched records ทิ้งทันที ถ้าอยากเก็บไว้ตรวจสอบควรใช้ `left join` หรือแยก `left_anti` เพื่อเช็กก่อน

3. **Join แล้วไม่เช็ก row count**

   ถ้า master table มี duplicate key อาจทำให้ rows เพิ่มขึ้นผิดปกติ และ aggregate ซ้ำ

4. **เลือก columns หลัง join ไม่ชัด**

   หลัง join อาจมีชื่อ column ซ้ำหรือ ambiguous ควรใช้ `.select()` ให้ชัดว่า output ต้องการ column ไหน

5. **สับสน `left_anti` กับ `left_semi`**

   - `left_anti` = records ที่ไม่ match
   - `left_semi` = records ที่ match แต่ไม่เอา columns จากขวาเข้ามา

6. **Aggregate หลัง join แบบไม่รู้ grain**

   ก่อน aggregate ต้องรู้ว่าแต่ละ row หมายถึงอะไร เช่น 1 row ต่อ transaction หรือ 1 row ต่อรายการย่อยใน transaction เดียวกัน (transaction line item) เพราะถ้า grain เปลี่ยน ตัวเลข summary ก็อาจเปลี่ยนตาม

7. **ใช้ `collect()` กับ summary ใหญ่เกินไป**

   ถึงจะเป็น summary แต่ถ้า output ยังใหญ่ ควรใช้ `show()` เพื่อ preview บางส่วน หรือ `write` / `saveAsTable()` เพื่อเขียนผลลัพธ์เป็น Lakehouse table แทนการดึงทั้งหมดกลับเข้า driver


## Expected Output / Success Criteria

เมื่อจบ Day 06 ควรทำได้:

- Aggregate transaction ด้วย `groupBy().agg()` ได้
- สรุปยอด transaction ต่อ customer และต่อ channel ได้
- ใช้ `inner join` เพื่อ enrich transaction กับ customer master ได้
- ใช้ `left join` เพื่อ enrich product โดยยังเก็บ unmatched product records ได้
- ใช้ `left_anti` หา unknown customer/product records ได้
- ใช้ `left_semi` filter เฉพาะ valid customer transactions ได้
- สร้าง `df_gold_customer_revenue` เป็น Gold-style summary ได้
- เขียนและอ่าน Lakehouse table `day06_customer_revenue_summary` ได้

## Final takeaway

Aggregation และ Join คือแกนหลักของการเปลี่ยน raw/cleaned data ให้กลายเป็น business-ready dataset

Mindset ที่สำคัญกว่า syntax คือ:

> ก่อน join ต้องรู้ grain และ join key  
> หลัง join ต้องเช็ก row count ว่าเพิ่มหรือลดผิดปกติไหม  
 
> ก่อน aggregate ต้องรู้ว่า records ไหนควรถูกนับ   
> หลัง aggregate ต้องเช็กว่าตัวเลข summary สอดคล้องกับ business logic หรือไม่

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day06_customer_revenue_summary")